# ANAADHI — Android/Kaggle Scene Still Generator

Phone-only GPU notebook for approved ANAADHI film shots.

**First production target:** `SC001_SH001`.

Run cells from top to bottom. Generate one shot, approve it, then continue.

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors peft

In [ ]:
import os, math, torch
from pathlib import Path
from PIL import Image
from diffusers import AutoPipelineForText2Image
from IPython.display import display
assert torch.cuda.is_available(), "GPU is not enabled. In Kaggle Notebook settings, enable a GPU accelerator."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

## Shot configuration

Edit only this cell when moving to another shot. The default is the locked **Scene 001 / Shot 001** establishing image.

In [ ]:
SHOT_ID = "SC001_SH001"
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
PROMPT = """cinematic photoreal feature-film still, pre-dawn Bhaigaara buffer forest in Karnataka Western Ghats,
a century of subtle unrecorded technological evolution, black monsoon rain falling through ancient trees,
massive wet roots around hidden sensor stakes, a raised rough timber cabin deep in the forest,
propellerless police drones shaped like black kites hovering high and discreetly,
Paraane medical transports concealed beneath areca-leaf camouflage,
distant restrained perimeter silhouettes only, no firefight,
Indian future production design grounded in Karnataka materials,
natural rain physics, volumetric mist, wet laterite earth, realistic wood and foliage,
dark psychological thriller, premium theatrical cinematography, anamorphic composition,
very wide establishing shot, low restrained camera, deep atmosphere, realistic scale,
no text in image, no borders, no baked black bars"""
NEGATIVE_PROMPT = """daylight, sunny sky, generic European forest, American police cars, cyberpunk neon city,
firefight, muzzle flash, explosion, fantasy armour, cartoon, anime, illustration,
storyboard boxes, captions, subtitles, watermark, logo text, letterbox bars,
bad architecture, duplicated cabin, warped trees, low resolution, blurry, oversaturated"""
SEED = 1001
STEPS = 32
GUIDANCE = 6.5
GEN_WIDTH = 1344
GEN_HEIGHT = 640
MASTER_WIDTH = 3840
MASTER_HEIGHT = 1608
REFERENCE_IMAGE = ""
REFERENCE_STRENGTH = 0.65
OUTPUT_DIR = Path("/kaggle/working/anaadhi_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Shot:", SHOT_ID)

## Load SDXL

The model is loaded in FP16. Model CPU offload reduces VRAM pressure on a single Kaggle GPU.

In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print("Model ready:", MODEL_ID)

## Optional uploaded visual reference

If `REFERENCE_IMAGE` points to a real image, the notebook loads the official SDXL IP-Adapter and uses the uploaded image as visual guidance. Leave it blank for text-only generation.

In [ ]:
reference = None
if REFERENCE_IMAGE and Path(REFERENCE_IMAGE).exists():
    reference = Image.open(REFERENCE_IMAGE).convert("RGB")
    pipe.load_ip_adapter(
        "h94/IP-Adapter",
        subfolder="sdxl_models",
        weight_name="ip-adapter_sdxl.bin",
    )
    pipe.set_ip_adapter_scale(REFERENCE_STRENGTH)
    print("Reference enabled:", REFERENCE_IMAGE)
    display(reference)
else:
    print("No reference image selected. Using screenplay prompt only.")

## Generate the shot

The generated working image is centre-cropped to the exact same ratio as the 3840×1608 ANAADHI master. No black bars are added.

In [ ]:
generator = torch.Generator(device="cpu").manual_seed(SEED)
kwargs = dict(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=GEN_WIDTH,
    height=GEN_HEIGHT,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    generator=generator,
)
if reference is not None:
    kwargs["ip_adapter_image"] = reference
image = pipe(**kwargs).images[0]
target_ratio = MASTER_WIDTH / MASTER_HEIGHT
new_h = round(image.width / target_ratio)
if new_h > image.height:
    new_w = round(image.height * target_ratio)
    x0 = (image.width - new_w) // 2
    scoped = image.crop((x0, 0, x0 + new_w, image.height))
else:
    y0 = (image.height - new_h) // 2
    scoped = image.crop((0, y0, image.width, y0 + new_h))
out_path = OUTPUT_DIR / f"{SHOT_ID}_seed{SEED}.png"
scoped.save(out_path)
print("Saved:", out_path)
print("Working output:", scoped.size, "ratio:", round(scoped.width / scoped.height, 4))
display(scoped)

## Approve / retry

If the shot is wrong, change only the necessary prompt detail or seed and rerun the **Generate the shot** cell.

Approved files remain in `/kaggle/working/anaadhi_outputs/` and can be downloaded from Kaggle's Output panel on Android.